In [1]:
import torch
from transformers import BertForMaskedLM, BertTokenizer

2025-04-17 18:32:13.169107: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744914733.181273     385 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744914733.185028     385 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-17 18:32:13.198362: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
import operator

In [2]:
def cuentaPalabras(texto):
    palabras = texto.split()
    print("Total de palabras:" + str(len(palabras)))
    #Definir arreglo de vocabulario
    vocabulario = {}

    # Analiza cada palabra para igresarla al vocabulario y si ya existe aumenta el contador para conocer su frecuencia
    for palabra in palabras:
        if palabra in vocabulario:
            vocabulario[palabra] += 1
        else:
            vocabulario[palabra] = 1

    return vocabulario
# Mostrar el diccionario resultante

In [3]:
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", do_lower_case=False)
model = BertForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
e = model.eval()

In [ ]:
# Now test it

#[CLS] Token de clasificación
#[SEP] Token de separación. Para separar dos frases.

# text = "[CLS] Para solucionar los [MASK] de Chile, el presidente debe [MASK] de inmediato. [SEP]"
text = "[CLS] Para solucionar los [MASK] de Chile, el presidente debe [MASK] de inmediato. [SEP]"
masked_indxs = (4,11)

#Genera los tokens del texto
tokens = tokenizer.tokenize(text)
print("Tokens: ",tokens)
#Convierte los tokens en una cantidad numerica
indexed_tokens = tokenizer.convert_tokens_to_ids(tokens)
print("Tokens ids: ",indexed_tokens)

tokens_tensor = torch.tensor([indexed_tokens])

predictions = model(tokens_tensor)[0]

for i,midx in enumerate(masked_indxs):
    idxs = torch.argsort(predictions[0,midx], descending=True)
    predicted_token = tokenizer.convert_ids_to_tokens(idxs[:5])
    print('MASK',i,':',predicted_token)

Tokens:  ['[CLS]', 'Para', 'solucionar', 'los', '[MASK]', 'de', 'Chile', ',', 'el', 'presidente', 'debe', '[MASK]', 'de', 'inmediato', '.', '[SEP]']
Tokens ids:  [4, 1830, 14450, 1065, 0, 1008, 5571, 1017, 1040, 3599, 1964, 0, 1008, 7515, 1009, 5]
MASK 0 : ['problemas', 'conflictos', 'asuntos', 'males', 'temas']
MASK 1 : ['renunciar', 'actuar', 'intervenir', 'regresar', 'asumir']


In [5]:
f = open('./data/prueba1.txt', "r")
textoArchivo = f.read()
f.close()
vocabulario=cuentaPalabras(textoArchivo)
print("prueba1 Total de palabras no repetidas:" + str(len(set(vocabulario))))


Total de palabras:141
prueba1 Total de palabras no repetidas:29


In [8]:
valores_ord = sorted(vocabulario.items(), key=operator.itemgetter(1), reverse=True)
#imprimir la 100 palabras con mayor número de repeticiones
print(valores_ord[0:100])

[('de', 20), ('la', 15), ('a', 10), ('El', 5), ('cuarteto', 5), ('misión', 5), ('debe', 5), ('ser', 5), ('ciudad', 5), ('Houston,', 5), ('donde', 5), ('un', 5), ('45', 5), ('días', 5), ('para', 5), ('gravedad', 5), ('trasladado', 4), ('completarán', 4), ('programa', 4), ('rehabilitación', 4), ('readaptarse', 4), ('terrestre.', 4), ('llevado', 1), ('finalizarán', 1), ('plan', 1), ('recuperación', 1), ('reacostumbrarse', 1), ('del', 1), ('planeta.', 1)]


In [9]:
print(textoArchivo)

El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser llevado a la ciudad de Houston, donde finalizarán un plan de recuperación de 45 días para reacostumbrarse a la gravedad del planeta.


In [10]:
textoArchivo=textoArchivo.replace("planeta","[MASK]")

In [11]:
print(textoArchivo)

El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser llevado a la ciudad de Houston, donde finalizarán un plan de recuperación de 45 días para reacostumbrarse a la gravedad del [MASK].


In [15]:
# Now test it

# text = "[CLS] Para solucionar los [MASK] de Chile, el presidente debe [MASK] de inmediato. [SEP]"
# text = "[CLS] "+textoArchivo+" [SEP]"
text = textoArchivo
print(text)
masked_indxs = (1,11)

tokens = tokenizer.tokenize(text)
indexed_tokens = tokenizer.convert_tokens_to_ids(tokens)
tokens_tensor = torch.tensor([indexed_tokens])

predictions = model(tokens_tensor)[0]

for i,midx in enumerate(masked_indxs):
    idxs = torch.argsort(predictions[0,midx], descending=True)
    predicted_token = tokenizer.convert_ids_to_tokens(idxs[:5])
    print('MASK',i,':',predicted_token)

El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser trasladado a la ciudad de Houston, donde completarán un programa de rehabilitación de 45 días para readaptarse a la gravedad terrestre.
El cuarteto de la misión debe ser llevado a la ciudad de Houston, donde finalizarán un plan de recuperación de 45 días para reacostumbrarse a la gravedad del [MASK].
MASK 0 : ['cuar', 'sep', 'bu', 'quin', 'cuarto']
MASK 1 : ['ciudad', 'sede', 'localidad', 'oficina', 'ciudades']
